# Parsing a Dataset of Books Declared "Extremist" in Belarus

This notebook transforms a list of books that have been declared "extremist materials" by Belarusian authorities into a structured dataset suitable for analysis. The source of the list is [PEN Belarus website](https://bannedbooks.penbelarus.org/extremist_list/).

Each record initially consists of a single line containing the author(s), title, publisher, publication year, and the date on which the book was added to the extremist materials list. The notebook extracts these fields, standardizes the text, and identifies the language of each book title.

In [48]:
import pandas as pd
import re

text = """
1. Катерина Андреева, Игорь Ильяш, «Белорусский Донбасс» (Folio, 2020) 26.03.2021
2. Виктор Ляхор, «Военная история Беларуси. Герои. Символы. Цвета» (Минск: «Харвест», 2019) 18.04.2022
3. «Беларусь на распутье». Сборник статей / составитель и редактор А. Е. Тарас (Рига: ИБИК, 2017) 13.05.2022
4. Альгерд Бахарэвіч, «Сабакі Эўропы» 17.05.2022
5. Зміцер Лукашук, Максім Гаруноў, «Беларуская нацыянальная iдэя» 17.05.2022
6. Константин Залесский, «Герои» Люфтваффе. Первая Персональная энциклопедия (Москва: «Яуза-пресс», 2014) 03.08.2022
7. Анатолий Тарас, «Краткий курс истории Беларуси IX-XXI вв.» (Минск: «Харвест», 2013) 12.09.2022
8. Святлана Казлова, «Аграрная палiтыка нацыстаў у Заходняй Беларусi: Планаванне. Забеспячэнне. Ажыццяўленне (1941-1944гг.)» (Мінск: «Янушкевіч», 2019) 07.10.2022
9. Іосіф Бродскі, «Балада пра маленькі буксір» (пер. з рус. Алесі Алейнік) (Мінск: «Янушкевіч», 2022) 18.10.2022
10. Уладзімір Арлоў, Павел Татарнікаў, «Айчына. Маляўнiчая гiсторыя. Ад Рагнеды да Касцюшкi» у 2 ч. Ч. 1 (Мінск: «Тэхналогія», 2019) 18.10.2022
11. Отто Скорцени, «Неизвестная война» (пер. с фр. О.П.Коваленок) (Минск: «Попурри», 2022) 18.11.2022
12. «Беларусь превыше всего». О национальной беларуской идее / составление, перевод, научное редактирование А. Е. Тарас (Смоленск: «Посох», 2011) 09.01.2023
13. Владимир Бешанов, «Год 1942 – учебный» (Минск: «Харвест», 2002) 09.01.2023
14. Вадим Деружинский, «Забытая Беларусь» (Минск: «Харвест», 2011) 09.01.2023
15. «Запiсы таварыства аматараў беларускай гiсторыi iмя Вацлава Ластоўскага. Выклiкi “рускага свету” i Беларусь». Выпуск. 6 / укладальнік і рэдактар А. Я. Тарас (Рыга: ІБГіК, 2016) 09.01.2023
16. Сергей Захаревич, «Большая кровь. Как СССР победил в войне 1941 – 1945 гг.» (Минск: «Современная школа», 2010) 09.01.2023
17. Леанід Лыч, «Нацыянальна-культурнае жыццё на Беларусi ў часы вайны (1941 – 1944 гг.)» (Вільня: «Наша будучыня», 2011) 09.01.2023
18.«Постсоветский транзит: между демократией и диктатурой». Сборник статей / составитель и редактор А. Е. Тарас (Рига: ИБИК, 2015) 09.01.2023
19. «Праблемы гуманiтарнай бяспекi Беларусi». Матэр’ялы навукова-практычнай канферэнцыi / пад рэдакцыяй А. Я. Тараса (Рыга: ІБГіК, 2013) 09.01.2023
20. Виктор Суворов, «Тень Победы» (Донецк: «Сталкер», 2003) 09.01.2023
21. Анатолий Тарас, «Страницы прошлого». Статьи по истории Беларуси (Минск: «Харвест», 2017) 09.01.2023
22. «Трансфармацыi ментальнасцi беларусаў у XXI ст.» Матэр’ялы навукова-практычнай канферэнцыi / Пад рэдакцыяй А. Я. Тараса (Рыга: ІБГіК, 2013) 09.01.2023
23. Олег Усачев, «Кто, как и зачем убил Вильгельма Кубе» (Минск: «Харвест», 2013) 09.01.2023
24. Аляксандр Смалянчук, «„Вызваленыя“ i заняволеныя». Польска-беларускае памежжа 1939-1941 гг. у дакумэнтах беларускiх архiваў (Мінск: «Зміцер Колас», 2021) 12.01.2023
25. Альгерд Бахарэвіч, «Апошняя кнiга пана А» (Прага – Мiнск: «Вясна» – «Янушкевiч», 2020) 14.03.2023
26. Анатоль Гатоўчыц, «Адысея капiтана БНР» (Гомель, 2019) 16.03.2023
27. Сергей Сюзев, «Двадцать лет рабства». Беларусь сегодня («Издательские решения», 2017) 13.04.2023
28. Алиция Дыбковская, Малгожата Жарын, Ян Жарын, «История Польши с древнейших времен до наших дней» (Варшава: «Научное издательство ПВН», 1995) 25.05.2023
29. «Polskie piesni patriotyczne» (Сборник патриотических песен) (Варшава, 2007) 25.05.2023
30. Лідзія Арабей. «Выбраныя творы (Серыя “Беларускі кнігазбор”)» (Мінск: «Кнігазбор», 2013) 15.08.2023
31. Ларыса Геніюш, «Выбраныя творы (Серыя “Беларускі кнігазбор”)» (Мінск: «Беларускі кнігазбор», 2000) 15.08.2023
32. Наталля Арсеннева, «Выбраныя творы (Серыя “Беларускі кнігазбор”)» (Мінск: «Беларускі кнігазбор», 2002) 15.08.2023
33. Уладзімір Някляеў, «Выбраныя творы (Серыя “Беларускі кнігазбор”)» (Мінск: «Кнігазбор», 2009) 15.08.2023
34. Прадмова Язэпа Янушкевіча да кнігі Вінцэнт Дунін-Марцінкевіч. «Выбраныя творы (Серыя “Беларускі кнігазбор”)» (Мінск: «Беларуская навука», 2019) 15.08.2023
35. Тэкст твора «Плывуць вятры» з кнігі Вінцэнт Дунін-Марцінкевіч. «Выбраныя творы (Серыя “Беларускі кнігазбор”)» (Мінск: «Беларуская навука», 2019) 15.08.2023
36. Тэкст твора «Гутарка старога дзеда» з кнігі Вінцэнт Дунін-Марцінкевіч. «Выбраныя творы (Серыя “Беларускі кнігазбор”)» (Мінск: «Беларуская навука», 2019) 15.08.2023
37. Александр Татаренко. Горькие годы: трагедия Западной Беларуси 1944-1954 (А.Н. Вараксин, 2017) 08.12.2023
38. Виктор Суворов, «Беру свои слова обратно». Вторая часть трилогии “Тень Победы” 29.12.2023
39. Джон Толанд, «Адольф Гитлер» (Москва: «ИнтерДайджест», 1993) 30.01.2024
40. Александр Бушков, «Россия, которой не было: загадки, версии, гипотезы» (Москва: «ОЛМА-ПРЕСС», 1999) 07.02.2024
41. Виктор Ляхор, «Военная символика белорусов. Знамена и мундиры» (Минск: «Харвест», 2014) 28.03.2024
42. Александр Татаренко, «Недозволенная память. Западная Беларусь в документах и фактах. 1921-1954» (2006 год) 05.04.2024
43. Виктор Суворов, «День “М”. Когда началась Вторая мировая война?» («Инлес», 1994) 15.04.2024 і 07.06.2024
44. Виктор Суворов, «Ледокол. Кто начал Вторую Мировую войну?» («Новое время», 1992) 15.04.2024 і 07.06.2024
45. Вацлаў Ластоўскі, «Выбраныя творы (Беларускі кнігазбор. Серыя I. Мастацкая літаратура)» (Мінск: «Беларускі кнігазбор», 1997) 30.04.2024 і 14.10.2024
46. Алесь Петрашкевіч, «Выбраныя творы (Беларускі кнігазбор. Серыя I. Мастацкая літаратура)» (Мінск: «Беларуская навука», 2017) 30.04.2024
47. Прадмова Язэпа Янушкевіча да кнігі Драматычныя творы, вершаваныя аповесцi i апавяданнi / Вiнцэнт Дунiн-Марцiнкевiч. Збор твораў. У 2 тамах, том 1 (Мінск: «Мастацкая літаратура», 2007) 24.05.2024
48. Павел Севярынец, «Каменнае сэрца» (Мінск, 2018) 20.06.2024
49. Павел Севярынец, «Люблю Беларусь: 200 фэномэнаў нацыянальнай ідэі» (Вільнюс, 2008) 20.06.2024
50. Павел Севярынец, «Беларусалім. Кніга другая. Сэрца сьвятла» (Днепр, 2020) 20.06.2024
51. Павал Нічыпарук, Ігар Мельнікаў, «Адысея Палешука» (Мінск: «Альфа-кніга», 2017) 23.08.2024
52. Ігар Мельнікаў, «Забыты корпус: гісторыя польскага войска на тэрыторыі Бабруйшчыны ў 1918, 1919–1920 гадах» (Мінск: «Альфа-кніга», 2018) 23.08.2024
53. Ігар Мельнікаў, «На “мяжы цывілізацый”. Старонкі гісторыі даваеннага савецка-польскага кардона ў Беларусі» (Мінск: «Альфа-кніга», 2020) 23.08.2024
54. Олег Курылев, «Боевые награды Третьего Рейха», иллюстрированная энциклопедия (Москва: «Эксмо», 2006) 12.12.2024
55. Йен Бакстер, «Секретные архивы СС. Западный и Восточный фронт» (Москва: «Эксмо», 2005) 12.12.2024
56. Михаил Фишман, «Преемник. История Бориса Немцова и страны, в которой он не стал президентом» (Москва: Corpus (АСТ), 2022) 17.12.2024
57. Захар Шыбека, «Нарыс гісторыі Беларусі (1795–2002)» (Мінск: «Энцыклапедыкс», 2003) 27.12.2024
58. Вадим Деружинский, «Тайны белорусской истории» (Минск: ФУАинформ, 2011) 10.01.2025
59. Альбом «БНР-100 сьвяткаванне 100-гадовага юбiлею ў Беларусi i ў замежжы» («Инбелкульт», 2019) 30.01.2025
60. Марат Гаравы, «НКВД забiваў у Курапатах…» («Змiцер Колас», 2019) 30.01.2025
61. Зміцер Дашкевіч, «Чарвяк» (Вільня, «Viešoji įstaiga Gudų informacijos centras», 2013) 11.02.2025
62. Сяргей Навумчык, «Дзевяноста трэці» («Радыё Свабода», 2018) 11.02.2025
63. Сяргей Навумчык, «Дзевяноста другі» («Радыё Свабода», 2017) 11.02.2025
64. Віктар Хурсік, «Кроў i попел Дражна: гiсторыі партызанскага злачынства» (Мінск: В. Хурсік, 2003) 06.03.2025
65. Сергей Захаревич, «Партизаны СССР: от мифов к реальности» (Вильня: «Наша будучыня», 2012) 06.03.2025
66. Михаил Пинчук, «Советские партизаны: мифы и реальность» (Вильня: «Наша будучыня», 2014) 06.03.2025
67. Павел Шеремет, Светлана Калинкина, «Случайный президент» (Санкт-Петербург: «Лимбус Пресс», 2004)* 18.03.2025
68. Пшэмыслаў Фенрых, Вінцук Вячорка, «Даведнік беларускага журналіста» (Шчэцін: «Centrum Szkoleniowe Fundacji Rozwoju Demokracji Lokalnej», 1999) 04.04.2025
69. Заико Леонид, Романчук Ярослав, «Беларусь на разломе» (Минск: «Центр Мизеса», 2008) 04.04.2025
70. Клёк Штучны, «Забойства на вулiцы Макаёнка» (Варшава, «Янушкевіч», 2024) 23.05.2025
71. Марсі Шор, «Украінская ноч: Вельмі асабістыя гісторыі рэвалюцыі» (Кракаў: «Gutenberg Publisher», 2024)  23.05.2025
72. Аляксандр Лукашук, «Там, дзе няма цемнаты: Радыё Свабода» (Прага: «Vesna», 2024) 23.05.2025
73. «Ідыёт самы настаяшчы…» (Менск-Варшава-Масква, «KONTRA-PRESS», 2001) 29.05.2025
74. Федор фон Бок, «Я стоял у ворот Москвы» (Москва: «Яуза-пресс», 2024(?)) 27.06.2025
75. Эрих фон Манштейн, «Утерянные победы» («Центрполиграф», 2023(?)) 24.07.2025
76. Эрвин Роммель, «Боевые операции в Северной Африке и на Западном фронте в Европе. 1940–1944» («Центрполиграф», 2023) 24.07.2025
77. Джин Шарп, «От диктатуры к демократии» («Скрипториум», 2020) 24.07.2025
78. «Нацыяналiзм у сьвеце. Мiнулае. Сучаснасьць. Пэрспэктывы» (Менск: «Голас краю», 2002) 28.07.2025
79. «Verbrannte Dorfer. Nationalsozialistische Verbrechen an der landlichen Bevolkerung in Polen und der Sowjetunion im Zweiten Weltkrieg» («Сожженные деревни. Преступления национал-социализма против сельского населения в Польше и Советском Союзе во время Второй мировой войны») («Metropol», 2024) 28.07.2025
80. Леанiд Лаўрэш, «Лябёдка Iваноўскiх» (Издательские решения, 2025) 28.08.2025
81. Райнер Цительманн, «Гитлер. Мировоззрение революционера» (Москва: «Социум», 2023(?)) 10.09.2025
82. Герман Гот, «Крах операции “Барбаросса”» (Москва: «Яуза-пресс», 2025) 10.09.2025
83. Вильгельм фон Лееб, Франц Гальдер «Ленинградский “Блицкриг” 1941-1942» (Москва: «Яуза-пресс», 2023) 10.09.2025
84. Тимоти Снайдер, «Про свободу» (Киев: «Наш формат», 2023(?)) 26.09.2025
85. Валянцін Акудовіч, «Код адсутнасці» («Логвінаў») 26.09.2025
86. Валянцін Акудовіч, «Трэба ўявіць Сізіфа шчаслівым» («Логвінаў») 26.09.2025
87. Валер Гапееў, «Вольнеры. Прадвесце» («Янушкевіч», 2023) 29.09.2025
88. Валер Гапееў, «Вольнеры. Бясконцы дзень» («Янушкевіч», 2024) 29.09.2025
89. Douglas E. Nash & Remy Spezzano, Kampfgruppe Mühlenkamp 5. SS-Panzer Division “Wiking”, Eastern Poland, July 1944 (2020) 29.09.2025
90. «Вот они, а вот мы. Белорусская поэзия и стихи солидарности», Сост. В. Коркунов. Посл. У. Вериной. (М.: Недовольный, 2021) 3.11.2025
91. Саша Філіпенка, «Чырвоны крыж» (Кракаў: «Gutenberg», 2025) 13.11.2025
92. Саша Філіпенка, «Слон» (Кракаў: «Gutenberg», 2025) 13.11.2025
93. Гейнц Гудериан, «Воспоминания немецкого генерала. Танковые войска Германии во Второй мировой войне. 1939-1945» («Центрполиграф», 2022(?)) 14.11.2025
94. Эліяш Барт «Джа: Легенда аб забраным сэрцы» («Тэхналогія», 2023) 18.02.2026
95. «Дневники сотрудника НКВД: документальное разоблачение сталинизма»,  сост., ред., коммент. Зеленкова А. 06.03.2026
96. Павел Антипов, «Куда-нибудь приезжать что-нибудь делать и уезжать» («Мяне Няма», 2025) 13.03.2026
97. Юрий Фельштинский, «Беларусь Натальи Радиной. Журналистка против диктатора» («ISIA Media Verlag», 2025) 13.03.2026
98. Макар, «Апошняе пакаленне» 02.04.2026
99. Каміла Цень, «Наступны прыпынак – смерць» 02.04.2026
100. Макс Шчур, «Там, дзе нас няма» 02.04.2026
101. Аляксандр Цвікевіч, «Гістарычныя працы. Том 1» 02.04.2026
102. «Мясцовыя выбары ў найноўшай палiтычнай гiсторыi Беларусi» (Менск: Аналiтычны грудок, 2003) 25.04.2026
103. «Ситуация с правами человека в Беларуси в 2008 году». Аналитический обзор (Минск, 2009) 25.04.2026
104. «Найноўшая гiсторыя беларускага парлямэнтарызму» (Менск: Аналiтычны грудок, 2005) 25.04.2026
105. «Зборнiк аналiтычных дакладаў» (Вiльня: Цэнтр падтрымкi выдавецкiх iнiцыятываў, 2009) 25.04.2026
"""

rows = []

for line in text.strip().split('\n'):
    m = re.match(r'^(\d+)\.\s*(.*?)\s+(\d{1,2}\.\d{1,2}\.\d{4})$', line)
    if m:
        rows.append({
            'id': int(m.group(1)),
            'book': m.group(2),
            'date': pd.to_datetime(m.group(3), dayfirst=True)
        })

df = pd.DataFrame(rows)

df

,id,book,date
0,1,"Катерина Андреева, Игорь Ильяш, «Белорусский Д...",2021-03-26
1,2,"Виктор Ляхор, «Военная история Беларуси. Герои...",2022-04-18
2,3,«Беларусь на распутье». Сборник статей / соста...,2022-05-13
3,4,"Альгерд Бахарэвіч, «Сабакі Эўропы»",2022-05-17
4,5,"Зміцер Лукашук, Максім Гаруноў, «Беларуская на...",2022-05-17
...,...,...,...
100,101,"Аляксандр Цвікевіч, «Гістарычныя працы. Том 1»",2026-04-02
101,102,«Мясцовыя выбары ў найноўшай палiтычнай гiстор...,2026-04-25
102,103,«Ситуация с правами человека в Беларуси в 2008...,2026-04-25
103,104,«Найноўшая гiсторыя беларускага парлямэнтарызм...,2026-04-25


In [49]:
print(df.shape)
df.tail()

(105, 3)


,id,book,date
100,101,"Аляксандр Цвікевіч, «Гістарычныя працы. Том 1»",2026-04-02
101,102,«Мясцовыя выбары ў найноўшай палiтычнай гiстор...,2026-04-25
102,103,«Ситуация с правами человека в Беларуси в 2008...,2026-04-25
103,104,«Найноўшая гiсторыя беларускага парлямэнтарызм...,2026-04-25
104,105,«Зборнiк аналiтычных дакладаў» (Вiльня: Цэнтр ...,2026-04-25


## Extracting bibliographic information

The original dataset stores each book as a single text string containing multiple pieces of information.

In this section, I use regular expressions to separate each record into structured variables, including:

- author(s);
- book title;
- publisher;
- publication year;
- date of inclusion in the extremist materials list.

Because the formatting is not perfectly consistent across all records, the parsing rules are designed to be flexible enough to handle small variations while preserving as much information as possible.

In [50]:
import re
import pandas as pd

def parse_book(text):
    result = {
        'authors': None,
        'title': None,
        'publisher': None
    }

    # the book name in quotation marks
    title_match = re.search(r'«([^»]+)»', text)
    if title_match:
        result['title'] = title_match.group(1)

        # everything before the name is considered the author
        authors = text[:title_match.start()].strip(' ,')
        result['authors'] = authors if authors else None

    # fron the last parenthesis
    pub_match = re.search(r'\(([^()]*)\)\s*$', text)
    if pub_match:
        result['publisher'] = pub_match.group(1)

    return pd.Series(result)

df[['authors', 'title', 'publisher']] = df['book'].apply(parse_book)

df.head()

,id,book,date,authors,title,publisher
0,1,"Катерина Андреева, Игорь Ильяш, «Белорусский Д...",2021-03-26,"Катерина Андреева, Игорь Ильяш",Белорусский Донбасс,"Folio, 2020"
1,2,"Виктор Ляхор, «Военная история Беларуси. Герои...",2022-04-18,Виктор Ляхор,Военная история Беларуси. Герои. Символы. Цвета,"Минск: «Харвест», 2019"
2,3,«Беларусь на распутье». Сборник статей / соста...,2022-05-13,NaN,Беларусь на распутье,"Рига: ИБИК, 2017"
3,4,"Альгерд Бахарэвіч, «Сабакі Эўропы»",2022-05-17,Альгерд Бахарэвіч,Сабакі Эўропы,NaN
4,5,"Зміцер Лукашук, Максім Гаруноў, «Беларуская на...",2022-05-17,"Зміцер Лукашук, Максім Гаруноў",Беларуская нацыянальная iдэя,NaN


In [51]:
df[df['title'].isna()]

,id,book,date,authors,title,publisher
36,37,Александр Татаренко. Горькие годы: трагедия За...,2023-12-08,NaN,NaN,"А.Н. Вараксин, 2017"
88,89,"Douglas E. Nash & Remy Spezzano, Kampfgruppe M...",2025-09-29,NaN,NaN,2020


## Cleaning and standardizing the extracted data

After parsing, I normalize the text to improve consistency across records.

This includes correcting common character substitutions, standardizing spelling where necessary, and cleaning the extracted fields so they can be analyzed more reliably.

In [52]:
for col in ['book', 'title', 'authors', 'publisher']:
    if col in df.columns:
        df[col] = (
            df[col]
            .str.replace('i', 'і', regex=False)
            .str.replace('I', 'І', regex=False)
        )

## Identifying the language of each book

The dataset does not contain an explicit language field, so I infer the language directly from the book titles.

As I didn't find a good external language detection library for detecting Belarusian, I use a rule-based approach tailored specifically to Belarusian and Russian. General-purpose language detectors often struggle because many titles are short, consist of proper names, or contain vocabulary shared between the two languages.

My approach combines several linguistic indicators:

- letters that are unique or highly characteristic of Belarusian, such as **ў** and **і**;
- Belarusian spelling and morphology that differ systematically from Russian;
- frequently occurring Belarusian lexical items;
- characteristic Russian spellings and vocabulary.

Each title is evaluated against these sets of linguistic markers. Titles containing predominantly Belarusian features are classified as Belarusian, while those matching Russian features are classified as Russian.

Although this approach is not perfect, it is transparent, reproducible, and better suited to this specialized dataset than a generic language identification model.

In [53]:
def detect_language(title):
    t = str(title).lower()

    bel_markers = [
        'ў', 'і', 'ьня', 'І', 'цц', 'дзь',
        'беларус', 'нацыян', 'гістор', 'твора',
        'кніга', 'выбраныя', 'вершы', 'аповесц',
        'забойства', 'вуліцы', 'прыпынак',
            'апошняя',
    'апошняе',
    'адысея',
    'плывуць',
    'вятры',
    'гутарка',
    'дзеда',
    'палешук',
    'прадвесце',
    'бясконцы',
    'дзень',
    'чырвоны',
    'крыж',
    'легенда',
    'сэрца',
    'сэрцы',
    'дзе',
    'няма',
    'пакаленне'
    ]

    if any(marker in t for marker in bel_markers):
        return 'belarusian'

    return 'russian'

df['language'] = df['book'].apply(detect_language)

## Reviewing the language classification

To verify that the language detection performs as expected, I inspect a random sample of titles together with their assigned language labels.

In [54]:
df[['title', 'language']].sample(20)

,title,language
51,Забыты корпус: гісторыя польскага войска на тэ...,belarusian
101,Мясцовыя выбары ў найноўшай палітычнай гісторы...,belarusian
6,Краткий курс истории Беларуси ІX-XXІ вв.,belarusian
25,Адысея капітана БНР,belarusian
41,Недозволенная память. Западная Беларусь в доку...,belarusian
20,Страницы прошлого,belarusian
59,НКВД забіваў у Курапатах…,belarusian
55,"Преемник. История Бориса Немцова и страны, в к...",russian
63,Кроў і попел Дражна: гісторыі партызанскага зл...,belarusian
31,Выбраныя творы (Серыя “Беларускі кнігазбор”),belarusian


## Inspecting Russian-language titles

This filtered view makes it easier to manually review titles that were classified as Russian and identify any possible misclassifications.

In [55]:
df[df['language'] == 'russian']

,id,book,date,authors,title,publisher,language
5,6,"Константин Залесский, «Герои» Люфтваффе. Перва...",2022-08-03,Константин Залесский,Герои,"Москва: «Яуза-пресс», 2014",russian
10,11,"Отто Скорцени, «Неизвестная война» (пер. с фр....",2022-11-18,Отто Скорцени,Неизвестная война,"Минск: «Попурри», 2022",russian
12,13,"Владимир Бешанов, «Год 1942 – учебный» (Минск:...",2023-01-09,Владимир Бешанов,Год 1942 – учебный,"Минск: «Харвест», 2002",russian
15,16,"Сергей Захаревич, «Большая кровь. Как СССР поб...",2023-01-09,Сергей Захаревич,Большая кровь. Как СССР победил в войне 1941 –...,"Минск: «Современная школа», 2010",russian
17,18,«Постсоветский транзит: между демократией и ди...,2023-01-09,NaN,Постсоветский транзит: между демократией и дик...,"Рига: ИБИК, 2015",russian
19,20,"Виктор Суворов, «Тень Победы» (Донецк: «Сталке...",2023-01-09,Виктор Суворов,Тень Победы,"Донецк: «Сталкер», 2003",russian
22,23,"Олег Усачев, «Кто, как и зачем убил Вильгельма...",2023-01-09,Олег Усачев,"Кто, как и зачем убил Вильгельма Кубе","Минск: «Харвест», 2013",russian
27,28,"Алиция Дыбковская, Малгожата Жарын, Ян Жарын, ...",2023-05-25,"Алиция Дыбковская, Малгожата Жарын, Ян Жарын",История Польши с древнейших времен до наших дней,"Варшава: «Научное издательство ПВН», 1995",russian
37,38,"Виктор Суворов, «Беру свои слова обратно». Вто...",2023-12-29,Виктор Суворов,Беру свои слова обратно,NaN,russian
38,39,"Джон Толанд, «Адольф Гитлер» (Москва: «ИнтерДа...",2024-01-30,Джон Толанд,Адольф Гитлер,"Москва: «ИнтерДайджест», 1993",russian


## Summarizing the results

Finally, I calculate the number of books assigned to each language. This provides an overview of the linguistic composition of the dataset and can be used in subsequent analysis of which types of literature have been designated as extremist.

In [56]:
df['language'].value_counts()

language
belarusian    77
russian       28
Name: count, dtype: int64